In [1]:
# Copyright 2025-2026 Arm Limited and/or its affiliates.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

# Ethos-U delegate flow example

This guide demonstrates the full flow for running a module on Arm Ethos-U55 using ExecuTorch.
Tested on Linux x86_64 and macOS aarch64. If something is not working for you, please raise a GitHub issue and tag Arm.

Before you begin:
1. (In a clean virtual environment with a compatible Python version) Install executorch using `./install_executorch.sh`
2. Install Arm cross-compilation toolchain and simulators using `./examples/arm/setup.sh --i-agree-to-the-contained-eula`

With all commands executed from the base `executorch` folder.



*Some scripts in this notebook produces long output logs: Configuring the 'Customizing Notebook Layout' settings to enable 'Output:scrolling' and setting 'Output:Text Line Limit' makes this more manageable*

## AOT Flow

The first step is creating the PyTorch module and exporting it. Exporting converts the python code in the module into a graph structure. The result is still runnable python code, which can be displayed by printing the `graph_module` of the exported program.  

In [2]:
import torch

class Add(torch.nn.Module):
    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return x + y

example_inputs = (torch.ones(1,1,1,1),torch.ones(1,1,1,1))

model = Add()
model = model.eval()
exported_program = torch.export.export(model, example_inputs)
graph_module = exported_program.module(check_guards=False)

_ = graph_module.print_readable()

class GraphModule(torch.nn.Module):
    def forward(self, x, y):
        x: "f32[1, 1, 1, 1]"; y: "f32[1, 1, 1, 1]"; 

        x, y, = fx_pytree.tree_flatten_spec(([x, y], {}), self._in_spec)
        # File: /tmp/ipykernel_97281/1177226286.py:5 in forward, code: return x + y
        add: "f32[1, 1, 1, 1]" = torch.ops.aten.add.Tensor(x, y);  x = y = None
        return pytree.tree_unflatten((add,), self._out_spec)



To run on Ethos-U the `graph_module` must be quantized using the `arm_quantizer`. Quantization can be done in multiple ways and it can be customized for different parts of the graph; shown here is the recommended path for the EthosUBackend. Quantization also requires calibrating the module with example inputs.

Again printing the module, it can be seen that the quantization wraps the node in quantization/dequantization nodes which contain the computed quanitzation parameters.

With the default passes for the Arm Ethos-U backend, assuming the model lowers fully to the Ethos-U, the exported program is composed of a Quantize node, Ethos-U custom delegate and a Dequantize node. In some circumstances, you may want to feed quantized input to the Neural Network straight away, e.g. if you have a camera sensor outputting (u)int8 data and keep all the arithmetic of the application in the int8 domain. For these cases, you can apply the `exir/passes/quantize_io_pass.py`. See the unit test in `backends/arm/test/passes/test_ioquantization_pass.py`for an example how to feed quantized inputs and obtain quantized outputs.


In [3]:
from executorch.backends.arm.ethosu import EthosUCompileSpec
from executorch.backends.arm.quantizer import (
    EthosUQuantizer,
    get_symmetric_quantization_config,
)
from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e, prepare_pt2e

# Create a compilation spec describing the target for configuring the quantizer
# Some args are used by the Arm Vela graph compiler later in the example. Refer to Arm Vela documentation for an
# explanation of its flags: https://gitlab.arm.com/artificial-intelligence/ethos-u/ethos-u-vela/-/blob/main/OPTIONS.md
compile_spec = EthosUCompileSpec(
            target="ethos-u85-256",
            system_config="Ethos_U85_SYS_DRAM_Mid",
            memory_mode="Shared_Sram",
            extra_flags=["--output-format=raw", "--debug-force-regor"]
        )

# Create and configure quantizer to use a symmetric quantization config globally on all nodes
quantizer = EthosUQuantizer(compile_spec)
operator_config = get_symmetric_quantization_config()
quantizer.set_global(operator_config)

# Post training quantization
quantized_graph_module = prepare_pt2e(graph_module, quantizer)
quantized_graph_module(*example_inputs) # Calibrate the graph module with the example input
quantized_graph_module = convert_pt2e(quantized_graph_module)

_ = quantized_graph_module.print_readable()

# Create a new exported program using the quantized_graph_module
quantized_exported_program = torch.export.export(quantized_graph_module, example_inputs)

class GraphModule(torch.nn.Module):
    def forward(self, x, y):
        x: "f32[1, 1, 1, 1]"; y: "f32[1, 1, 1, 1]"; 

        x, y, = fx_pytree.tree_flatten_spec(([x, y], {}), self._in_spec)
        # No stacktrace found for following nodes
        quantize_per_tensor_default = torch.ops.quantized_decomposed.quantize_per_tensor.default(x, 0.003921568859368563, -128, -128, 127, torch.int8);  x = None

        # File: /tmp/ipykernel_97281/1177226286.py:5 in forward, code: return x + y
        dequantize_per_tensor_default = torch.ops.quantized_decomposed.dequantize_per_tensor.default(quantize_per_tensor_default, 0.003921568859368563, -128, -128, 127, torch.int8);  quantize_per_tensor_default = None

        # No stacktrace found for following nodes
        quantize_per_tensor_default_1 = torch.ops.quantized_decomposed.quantize_per_tensor.default(y, 0.003921568859368563, -128, -128, 127, torch.int8);  y = None

        # File: /tmp/ipykernel_97281/1177226286.py:5 in forward, code: return

The lowering in the EthosUBackend happens in five steps:

1. **Lowering to core Aten operator set**: Transform module to use a subset of operators applicable to edge devices. 
2. **Partitioning**: Find subgraphs which are supported for running on Ethos-U
3. **Lowering to TOSA compatible operator set**: Perform transforms to make the Ethos-U subgraph(s) compatible with TOSA 
4. **Serialization to TOSA**: Compiles the graph module into a TOSA graph 
5. **Compilation to NPU**: Compiles the TOSA graph into an EthosU command stream using the Arm Vela graph compiler. This makes use of the `compile_spec` created earlier.
Step 5 also prints a Network summary for each processed subgraph.

All of this happens behind the scenes in `to_edge_transform_and_lower`. Printing the graph module shows that what is left in the graph is two quantization nodes for `x` and `y` going into an `executorch_call_delegate` node, followed by a dequantization node.

In [4]:
from executorch.backends.arm.ethosu import EthosUPartitioner
from executorch.exir import (
    EdgeCompileConfig,
    ExecutorchBackendConfig,
    to_edge_transform_and_lower,
)
from executorch.extension.export_util.utils import save_pte_program

# Create partitioner from compile spec
partitioner = EthosUPartitioner(compile_spec)

# Lower the exported program to the Ethos-U backend
edge_program_manager = to_edge_transform_and_lower(
            quantized_exported_program,
            partitioner=[partitioner],
            compile_config=EdgeCompileConfig(
                _check_ir_validity=False,
            ),
        )

# Convert edge program to executorch
executorch_program_manager = edge_program_manager.to_executorch(
            config=ExecutorchBackendConfig(extract_delegate_segments=False)
        )

_ = executorch_program_manager.exported_program().module(check_guards=False).print_readable()

# Save pte file
save_pte_program(executorch_program_manager, "ethos_u_minimal_example.pte")

/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)



Network summary for out
Accelerator configuration               Ethos_U85_256
System configuration             Ethos_U85_SYS_DRAM_Mid
Memory mode                               Shared_Sram
Accelerator clock                                1000 MHz
Design peak SRAM bandwidth                      29.80 GB/s

Total SRAM used                                  0.05 KiB

CPU operators = 0 (0.0%)
NPU operators = 4 (100.0%)

Average SRAM bandwidth                           0.89 GB/s
Input   SRAM bandwidth                           0.00 MB/batch
Weight  SRAM bandwidth                           0.00 MB/batch
Output  SRAM bandwidth                           0.00 MB/batch
Total   SRAM bandwidth                           0.00 MB/batch
Total   SRAM bandwidth            per input      0.00 MB/inference (batch size 1)

Neural network macs                                 3 MACs/batch

class GraphModule(torch.nn.Module):
    def forward(self, x, y):
        x: "f32[1, 1, 1, 1]"; y: "f32[1, 1, 1, 1]"; 

  

/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


'ethos_u_minimal_example.pte'

## Build executor runtime

After the AOT compilation flow is done, the runtime can be cross compiled and linked to the produced .pte-file using the Arm cross-compilation toolchain. This is done in two steps:
1. Build and install the executorch libraries and EthosUDelegate.
2. Build and link the `arm_executor_runner` and generate kernel bindings for any non delegated ops.

In [ ]:
%%bash

cd ../executorch/examples/arm

# Ensure the arm-none-eabi-gcc toolchain and FVP:s are available on $PATH
source arm-scratch/setup_path.sh

# Build executorch libraries cross-compiled for arm baremetal to executorch/cmake-out-arm
cmake --preset arm-baremetal \
-DCMAKE_BUILD_TYPE=Release \
-B../../cmake-out-arm ../..
cmake --build ../../cmake-out-arm --target install -j$(nproc) 

/home/ubuntu/learn_executorch_on_arm


In [ ]:
%%bash 

cd ../executorch/examples/arm

source arm-scratch/setup_path.sh

# Build example executor runner application to examples/arm/ethos_u_minimal_example
cmake -DCMAKE_TOOLCHAIN_FILE=$(pwd)/ethos-u-setup/arm-none-eabi-gcc.cmake \
      -DCMAKE_BUILD_TYPE=Release \
      -DET_PTE_FILE_PATH=ethos_u_minimal_example.pte \
      -DTARGET_CPU=cortex-m85 \
      -DETHOSU_TARGET_NPU_CONFIG=ethos-u85-256 \
      -DSYSTEM_CONFIG=Ethos_U85_SYS_DRAM_Mid \
      -Bethos_u_minimal_example \
      executor_runner
cmake --build ethos_u_minimal_example -j$(nproc) -- arm_executor_runner

-- Fetching Ethos-U content into /home/ubuntu/executorch/examples/arm/executor_runner/../../../examples/arm/arm-scratch/ethos-u
'bash' '-c' 'pwd && source backends/arm/scripts/utils.sh && patch_repo /home/ubuntu/executorch/examples/arm/executor_runner/../../../examples/arm/arm-scratch/ethos-u 24950bd4381b6c51db0349a229f8ba86b8e1093f /home/ubuntu/executorch/examples/arm/executor_runner/../../../examples/arm/ethos-u-setup'
/home/ubuntu/executorch
[patch_repo] Patching ethos-u repo_dir:/home/ubuntu/executorch/examples/arm/executor_runner/../../../examples/arm/arm-scratch/ethos-u base_rev:24950bd4381b6c51db0349a229f8ba86b8e1093f patch_dir:/home/ubuntu/executorch/examples/arm/executor_runner/../../../examples/arm/ethos-u-setup/ethos-u
~/executorch/examples/arm/arm-scratch/ethos-u ~/executorch
HEAD is now at 24950bd 25.05 Release
Applying: Remove unused projects from 25.05 manifest
[patch_repo] Patched ethos-u @ tags/25.05-1-gfb46d5d in /home/ubuntu/executorch/examples/arm/executor_runner/..

Configuring target corstone-320


-- *******************************************************
-- PROJECT_NAME                           : ethosu_core_driver
-- ETHOSU_TARGET_NPU_CONFIG               : ethos-u85-256
-- CMAKE_SYSTEM_PROCESSOR                 : cortex-m55
-- CMSIS_PATH                             : /home/ubuntu/executorch/examples/arm/arm-scratch/ethos-u/core_software/cmsis_6
-- ETHOSU_LOG_ENABLE                      : ON
-- ETHOSU_LOG_SEVERITY                    : warning
-- ETHOSU_INFERENCE_TIMEOUT               : Default (no timeout)
-- *******************************************************
-- *******************************************************
-- PROJECT_NAME                           : core_software
-- CORE_SOFTWARE_RTOS                     : All
-- CORE_SOFTWARE_ACCELERATOR              : NPU
-- CORE_LOG_ENABLE                        : ON
-- CORE_LOG_SEVERITY                      : warning
-- *******************************************************
-- *********************************************

Skipping driver_unit_conv application
Skipping FreeRTOS application
Skipping ThreadX application
Skipping message handler openamp


-- *******************************************************
-- PROJECT_NAME                           : ethos-u-corstone-320
-- TR_ARENA_SIZE                          : 
-- MESSAGE_HANDLER_ARENA_SIZE             : 
-- *******************************************************
-- Could NOT find tokenizers (missing: tokenizers_DIR)


...

[ 95%] Building CXX object CMakeFiles/arm_executor_runner.dir/home/ubuntu/executorch/examples/arm/arm-scratch/ethos-u/core_platform/drivers/mpu/src/mpu.cpp.obj


<command-line>: warning: "ETHOSU_ARENA" redefined
<command-line>: note: this is the location of the previous definition


[100%] Linking CXX executable arm_executor_runner


/home/ubuntu/executorch/examples/arm/arm-scratch/arm-gnu-toolchain-13.3.rel1-aarch64-arm-none-eabi/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: warning: section `.ddr' type changed to PROGBITS
/home/ubuntu/executorch/examples/arm/arm-scratch/arm-gnu-toolchain-13.3.rel1-aarch64-arm-none-eabi/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: arm_executor_runner: section `.sram' can't be allocated in segment 2
LOAD: .ddr .sram
/home/ubuntu/executorch/examples/arm/arm-scratch/arm-gnu-toolchain-13.3.rel1-aarch64-arm-none-eabi/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: warning: arm_executor_runner has a LOAD segment with RWX permissions


[100%] Built target arm_executor_runner


# Run on simulated model

We can finally use the `backends/arm/scripts/run_fvp.sh` utility script to run the .elf-file on simulated Arm hardware. The example application is by default built with an input of ones, so the expected result of the quantized addition should be close to 2.

In [ ]:
%%bash 

cd ../executorch/examples/arm

source arm-scratch/setup_path.sh

# Run the example
../../backends/arm/scripts/run_fvp.sh --elf=ethos_u_minimal_example/arm_executor_runner --target=ethos-u85-256

--------------------------------------------------------------------------------
Running /home/ubuntu/executorch/examples/arm/ethos_u_minimal_example/arm_executor_runner for ethos-u85-256 run with FVP:FVP_Corstone_SSE-320 num_macs:256 timeout:600
--------------------------------------------------------------------------------


telnetterminal0: Listening for serial connection on port 5000
telnetterminal1: Listening for serial connection on port 5001
telnetterminal2: Listening for serial connection on port 5002
telnetterminal5: Listening for serial connection on port 5003
I [executorch:arm_executor_runner.cpp:1274 main()] PTE @ 0x70200000 [----ET12]
I [executorch:arm_executor_runner.cpp:602 runner_init()] PTE Model data loaded. Size: 3456 bytes.
I [executorch:arm_executor_runner.cpp:618 runner_init()] Model buffer loaded, has 1 methods
I [executorch:arm_executor_runner.cpp:628 runner_init()] Running method forward
I [executorch:arm_executor_runner.cpp:639 runner_init()] Setup Method allocator pool. Size: 62914560 bytes.
I [executorch:arm_executor_runner.cpp:655 runner_init()] Setting up planned buffer 0, size 48.
I [executorch:EthosUBackend.cpp:81 init()] data:0x70200070
I [executorch:arm_executor_runner.cpp:764 runner_init()] Method 'forward' loaded.
I [executorch:arm_executor_runner.cpp:766 runner_init()] Pr